# 2_models/06 — VTE: text risk score vs. Khorana (CPU, cluster)

Runs the whole VTE / Khorana comparison end to end, from the endpoint build to the
figure-data CSVs. Run it in a compute-backed Jupyter CPU session after `1_data/01_preprocessing.ipynb`
(cohort, labs, covariates) and the embedding knit in `1_data/03_prediction_datasets.ipynb`.

Stages, in order:
1. **Endpoints** (`pipelines.preprocessing.generate_embedding_prediction_datasets`), once per
   anchor. Writes the `vte` scheme's `vte_embedding_prediction_df*.parquet`. It has two events:
   - `vte`: curated acute PE/DVT ICD-10 codes, the primary endpoint (`shared/vte.py`).
   - `vte_phecode`: PE/DVT phecodes 415.1x and 452.2x, a sensitivity endpoint.

   Patients with any VTE code on or before the anchor have null events and drop out.
   **The generator rewrites every scheme's prediction dataset for that anchor, not just `vte`.**
2. **Khorana inputs** (`pipelines.preprocessing.build_published_scores`) for each anchor and each
   lab window (30 and 90 days). It reads labs, BMI (HEALTH_HISTORY) and ESA orders (MEDICATIONS),
   and writes `published_scores_df*.csv.gz` plus a coverage CSV to `FEATURE_PATH`.
   - A read-only audit runs first.
   - Note scores are extracted only if missing; the builder needs them for the OS scores'
     ECOG items, though Khorana does not use them.
3. **Text model** for scheme `vte`, for each anchor and event:
   - `run_full_cohort_event`: 5-fold CV, text vs. base model.
   - `run_full_cohort_risk_scores`: out-of-fold `text_risk_scores.csv`.

   Both runners skip outputs that already exist unless `OVERWRITE`.
4. **Comparison** (`figures.prep.khorana_vte`). Compares Khorana, text, and Khorana+text C-index
   on shared comparable pairs, for full follow-up and for a 180-day horizon. Writes
   `khorana_vte_{cindex,delta,cox,km,cohort}.csv` to `FIGURE_DATA_DIR`.

Each stage runs as a `python -m` subprocess with `check=True`, so a failure stops the notebook at
that stage. The stage-1 to stage-3 cells can be skipped individually with the `RUN_*` flags
once their outputs exist.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import polars as pl  # noqa: E402

from config import FEATURE_PATH, FIGURE_DATA_DIR, SURV_PATH  # noqa: E402
from schemes import embedding_file, full_cohort_event_dir, full_cohort_risk_dir  # noqa: E402


def run_stage(args: list[str], env: dict | None = None) -> None:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    started = time.perf_counter()
    subprocess.run([sys.executable, "-m", *args], cwd=REPO_ROOT, check=True,
                   env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})})
    print(f"--- done in {(time.perf_counter() - started) / 60:.1f} min", flush=True)


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

The anchors, lab windows and events default to the run grid of `figures.prep.khorana_vte`. The
comparison stage reads all of them, and any it cannot find are written as `missing_inputs` rows.

In [ ]:
from figures.prep.khorana_vte import ANCHORS_TO_RUN, EVENTS_TO_RUN, LAB_WINDOWS_TO_RUN, VTE_SCHEME

ANCHORS = list(ANCHORS_TO_RUN)          # ("treatment", "sequencing")
LAB_WINDOWS = list(LAB_WINDOWS_TO_RUN)  # (30, 90) days; 30 is the builder's default
EVENTS = list(EVENTS_TO_RUN)            # ("vte", "vte_phecode")

RUN_ENDPOINTS = True      # stage 1; False to reuse existing prediction datasets
RUN_SCORES = True         # stage 2
RUN_TRAINING = True       # stage 3
OVERWRITE = False         # stage 3: refit even when the runner's outputs exist

N_JOBS = max(1, (os.cpu_count() or 1) - 1)  # CV jobs per training task (tasks run one at a time)
MAX_ITER = 1000
BACKEND = "threading"

N_BOOT = 1000             # stage 4 bootstrap (primary treatment / 30-day / vte runs only)
PREP_N_JOBS = None        # stage 4 worker processes; None = auto

# One training task owns the box at a time, so let joblib take the cores and keep BLAS at one
# thread per job to avoid oversubscription.
TRAIN_ENV = {var: "1" for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                                  "NUMEXPR_NUM_THREADS")}

print(f"scheme:      {VTE_SCHEME}")
print(f"anchors:     {ANCHORS}")
print(f"lab windows: {LAB_WINDOWS}")
print(f"events:      {EVENTS}")
print(f"n_jobs:      {N_JOBS}")

## 1. VTE endpoints

Rebuilds the prediction datasets for each anchor, then summarizes the `vte` scheme's events.
`excluded` counts patients with prior VTE, whose event is null. Before training, check that
the event rate is plausible (a few percent) and that there are at least ~100 events.

In [ ]:
if RUN_ENDPOINTS:
    for anchor in ANCHORS:
        run_stage(["pipelines.preprocessing.generate_embedding_prediction_datasets", "--anchor", anchor])
else:
    print("Stage 1 skipped (RUN_ENDPOINTS = False); reading existing datasets.")

rows = []
for anchor in ANCHORS:
    path = Path(SURV_PATH) / embedding_file(VTE_SCHEME, anchor)
    if not path.is_file():
        print(f"[MISSING] {path}")
        continue
    frame = pl.read_parquet(path, columns=[c for e in EVENTS for c in (e, f"tt_{e}")])
    for event in EVENTS:
        kept = frame.filter(pl.col(event).is_not_null() & pl.col(f"tt_{event}").is_not_null())
        rows.append({
            "anchor": anchor, "event": event, "n_patients": frame.height,
            "excluded": frame.height - kept.height, "n_at_risk": kept.height,
            "n_events": int(kept[event].sum()),
            "event_rate": kept[event].mean(),
            "median_tt_event_days": kept.filter(pl.col(event) == 1)[f"tt_{event}"].median(),
        })

with pl.Config(tbl_rows=20, float_precision=3):
    print(pl.DataFrame(rows))

## 2. Khorana inputs

The audit is read-only. Review its `CANCER_GROUP` table: Khorana's site points and eligibility
key on LEUKEMIA, MYELOMA, STOMACH, PANCREAS, GERM_CELL, and so on. The next cell checks the two
strings the audit does not cover:
- the BMI `HEALTH_HISTORY_TYPE` (the builder matches exactly `"BMI"`);
- the medication names matched by the ESA regex.

If either looks wrong, fix `profile_sources.HH_BMI_TYPE` or `build_published_scores.ESA_NAME_PATTERN`
before running the builder. The builder runs once for each (anchor, lab window). Each run rewrites
that combination's whole published-score frame, including the OS scores.

In [ ]:
from pipelines.preprocessing import profile_sources as ps
from pipelines.preprocessing.build_published_scores import ESA_NAME_PATTERN

if RUN_SCORES:
    run_stage(["pipelines.preprocessing.audit_published_score_inputs", "--anchor", ANCHORS[0]])

hh_types = (ps.load_health_history([ps.HH_TYPE]).group_by(ps.HH_TYPE).len()
            .filter(pl.col(ps.HH_TYPE).str.contains(r"(?i)bmi|body mass|weight|height"))
            .sort("len", descending=True).collect())
esa_names = (ps.load_medications([ps.MEDS_NCI_NM, ps.MEDS_OTHER_NM])
             .filter(pl.col(ps.MEDS_NCI_NM).str.contains(ESA_NAME_PATTERN)
                     | pl.col(ps.MEDS_OTHER_NM).str.contains(ESA_NAME_PATTERN))
             .group_by(ps.MEDS_NCI_NM, ps.MEDS_OTHER_NM).len().sort("len", descending=True).collect())

with pl.Config(tbl_rows=30, fmt_str_lengths=60):
    print(f"HEALTH_HISTORY types like BMI (builder uses {ps.HH_BMI_TYPE!r}):")
    print(hh_types)
    print("\nMedication names matching the ESA pattern:")
    print(esa_names)

In [ ]:
from pipelines.preprocessing.extract_note_scores import note_score_path

if RUN_SCORES:
    for anchor in ANCHORS:
        if os.path.exists(note_score_path(anchor)):
            print(f"[reuse] note scores for {anchor}: {note_score_path(anchor)}")
        else:
            run_stage(["pipelines.preprocessing.extract_note_scores", "--anchor", anchor])
    for anchor in ANCHORS:
        for window in LAB_WINDOWS:
            run_stage(["pipelines.preprocessing.build_published_scores", "--anchor", anchor,
                       "--lab-window-days", str(window)])
else:
    print("Stage 2 skipped (RUN_SCORES = False); reading existing coverage.")

coverage = []
for anchor in ANCHORS:
    suffix = "" if anchor == "treatment" else f"__{anchor}"
    for window in LAB_WINDOWS:
        lab_suffix = "" if window == 30 else f"__lab{window}d"
        path = Path(FEATURE_PATH) / f"published_scores_coverage{suffix}{lab_suffix}.csv.gz"
        if not path.is_file():
            print(f"[MISSING] {path}")
            continue
        coverage.append(pl.read_csv(path).filter(pl.col("score") == "khorana")
                        .select(pl.lit(anchor).alias("anchor"), pl.lit(window).alias("lab_window_days"),
                                "n_eligible", "n_complete"))

if coverage:
    print(pl.concat(coverage).with_columns(
        (pl.col("n_complete") / pl.col("n_eligible")).round(3).alias("frac_complete")))

## 3. Text model for the `vte` scheme

For each (anchor, event), the full-cohort CV runs first and then the out-of-fold risk scores,
which read the CV's best hyperparameters from `text_val.csv`. Tasks run one at a time, and each
gets `N_JOBS` cores. `run_full_cohort_event` raises if an event fails `validate_cox_inputs`
(for example, too few events). That stops the notebook here, and the message names the event.

In [ ]:
TRAIN_ARGS = ["--n-jobs", str(N_JOBS), "--max-iter", str(MAX_ITER), "--backend", BACKEND]
if OVERWRITE:
    TRAIN_ARGS.append("--overwrite")

if RUN_TRAINING:
    for anchor in ANCHORS:
        for event in EVENTS:
            task = ["--scheme", VTE_SCHEME, "--event", event, "--anchor", anchor, *TRAIN_ARGS]
            run_stage(["pipelines.training.run_full_cohort_event", *task], env=TRAIN_ENV)
            run_stage(["pipelines.training.run_full_cohort_risk_scores", *task], env=TRAIN_ENV)
else:
    print("Stage 3 skipped (RUN_TRAINING = False).")

perf = []
for anchor in ANCHORS:
    for event in EVENTS:
        event_dir = Path(full_cohort_event_dir(VTE_SCHEME, event, anchor))
        row = {"anchor": anchor, "event": event,
               "risk_scores": (Path(full_cohort_risk_dir(VTE_SCHEME, event, anchor))
                               / "text_risk_scores.csv").is_file()}
        for model in ("base", "text"):
            path = event_dir / f"{model}_test.csv"
            row[f"{model}_c_index"] = pl.read_csv(path)["mean_c_index"][0] if path.is_file() else None
        perf.append(row)

with pl.Config(tbl_rows=20, float_precision=3):
    print(pl.DataFrame(perf))

## 4. Khorana vs. text comparison

Writes the five `khorana_vte_*.csv` tables. Each row is keyed by anchor, lab window, event and
horizon. The tables shown below are restricted to the primary analysis: treatment anchor,
30-day window, `vte`. `unblocked_published_cindex` in the cohort table is the figure comparable
to Khorana validation studies, which report about 0.6. The blocked C-indices in
`khorana_vte_cindex.csv` are computed only within the text model's outer folds, so they can
differ from it.

In [ ]:
run_stage(["figures.prep.khorana_vte", "--n-boot", str(N_BOOT),
           *([] if PREP_N_JOBS is None else ["--n-jobs", str(PREP_N_JOBS)])])

PRIMARY = (pl.col("anchor") == "treatment") & (pl.col("lab_window_days") == 30) & (pl.col("event") == "vte")
results = {name: pl.read_csv(Path(FIGURE_DATA_DIR) / f"khorana_vte_{name}.csv")
           for name in ("cohort", "cindex", "delta")}

with pl.Config(tbl_rows=40, tbl_cols=20, float_precision=3, fmt_str_lengths=40):
    print("Cohort — every run")
    print(results["cohort"].select("anchor", "lab_window_days", "event", "horizon_days", "n_eligible",
                                   "n_complete", "n_patients", "n_events", "unblocked_published_cindex",
                                   "underpowered"))
    print("\nC-index — primary")
    print(results["cindex"].filter(PRIMARY).select("horizon_days", "model", "cindex", "ci_lower", "ci_upper",
                                                   "n_patients", "n_events", "status"))
    print("\nDelta C-index — primary")
    print(results["delta"].filter(PRIMARY).select("horizon_days", "model", "reference", "delta_cindex",
                                                  "ci_lower", "ci_upper"))